In [157]:
pip install langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [158]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

## Install Libraries

In [160]:
pip install youtube-transcript-api langchain langchain-community langchain-groq faiss-cpu tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [161]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion) Using Loader

In [163]:
import sys
!{sys.executable} -m pip uninstall youtube-transcript-api -y
!{sys.executable} -m pip install youtube-transcript-api --upgrade --no-cache-dir

Found existing installation: youtube-transcript-api 1.2.4
Uninstalling youtube-transcript-api-1.2.4:
  Successfully uninstalled youtube-transcript-api-1.2.4


In [164]:
from youtube_transcript_api import FetchedTranscript
print("Now working")

Now working


In [165]:
from langchain_community.document_loaders import YoutubeLoader

loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=Gfr50f6ZBvo&t=5s",
    add_video_info=False,
    language=["en", "hi"] 
)

docs = loader.load()

print(docs[0].page_content[:500])

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful 


## Step 1b - Indexing (Text Splitting)

In [167]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

print(chunks[:2])

[Document(metadata={'source': 'Gfr50f6ZBvo'}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai pr

In [168]:
len(chunks)

168

In [169]:
chunks[0]

Document(metadata={'source': 'Gfr50f6ZBvo'}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai pro

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [171]:
!pip install sentence-transformers
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4469.50it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [172]:
vector_store.index_to_docstore_id

{0: '38032ccb-3a0e-4c49-b3bc-e0dc7b684cc0',
 1: '43cfa0a4-5c8d-41c1-aa7b-3ae8543b7baa',
 2: 'aa5ca924-a2b4-403d-95f7-87abc0a8fe1c',
 3: 'fd13800d-141c-4d88-ac09-652c4fab75d0',
 4: 'a6f7e513-7950-49b8-a785-efeb1f21eea9',
 5: 'c0c7a794-eacb-4dab-b1a3-b1453a9ec8fb',
 6: 'b4156f98-2fe4-497d-9042-6c3309c12c44',
 7: 'fe9aeb40-1707-4ba3-9c21-86b981349f39',
 8: 'a82edd94-b699-41a6-b05e-ae934d23816a',
 9: '126b2f16-cee2-410b-b6b8-b1479d1a13fa',
 10: 'fce33a3d-6e83-4b84-af46-77c6bd3858b0',
 11: '2ce1c8d2-e85d-4c0b-b4ff-989404229629',
 12: 'ccc43d46-de94-42fe-b520-3563f270da74',
 13: '8c23d9fd-e5c7-4d47-afc7-e8e21368d1f5',
 14: '427b9550-08c4-4d38-8a01-581f958c57f2',
 15: 'a050dc74-96fc-4aee-b9a6-f22d272c3642',
 16: 'a5ffa7a0-ffde-4fc8-a974-c5dcba7882b4',
 17: '4e56751c-3c53-4ac8-8933-0a5d12ff5d52',
 18: 'fa75f755-7291-447b-bdfc-958d6063a325',
 19: '9f517b26-da02-4c13-9a95-03015a2fb52f',
 20: '3c600c1a-770d-4004-b8aa-94b96a9d6ec8',
 21: '2900e8b5-0aff-40b5-8565-965738cb88f7',
 22: 'afc4166a-c990-

In [173]:
vector_store.get_by_ids(['1c101669-1cf2-4c2e-9d36-a2eea3b81a4b'])

[]

## Step 2 - Retrieval

In [175]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [176]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000250C652C8C0>, search_kwargs={'k': 4})

In [177]:
retriever.invoke('What is deepmind')

[Document(id='33934f99-d42b-4178-90e1-fddfab331995', metadata={'source': 'Gfr50f6ZBvo'}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look

## Step 3 - Augmentation 

In [179]:
!pip install langchain-groq

In [180]:
from langchain_groq import ChatGroq

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=api_key
)
response = llm.invoke("Hello")
print(response.content)

Hello, how can I assist you today?


In [181]:
prompt = PromptTemplate(template = """
                                You are a helpful assistant.
                                Answer ONLY from the provided transcript context.
                                If the context is insufficient, just say you don't know.

                                {context}
                                Question: {question}
                                """,
                        input_variables = ['context', 'question']
                       )

In [182]:
question = "is the topic of aliens discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [183]:
retrieved_docs

[Document(id='d1221a51-59c6-4bb9-8014-8a593f84cbd3', metadata={'source': 'Gfr50f6ZBvo'}, page_content="space age we should have heard a cacophony of voices we should have joined that cacophony of voices and what we did we opened our ears and we heard nothing and many people who argue that there are aliens would say well we haven't really done exhaustive search yet and maybe we're looking in the wrong bands and and we've got the wrong devices and we wouldn't notice what an alien form was like to be so different to what we're used to but you know i'm not i don't really buy that that it shouldn't be as difficult as that like we i think we've searched enough there should be if it were everywhere if it was it should be everywhere we should see dyson's fears being put up sun's blinking in and out you know there should be a lot of evidence for those things and then there are other people argue well the sort of safari view of like well we're a primitive species still because we're not space fa

In [184]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [185]:
context_text

"space age we should have heard a cacophony of voices we should have joined that cacophony of voices and what we did we opened our ears and we heard nothing and many people who argue that there are aliens would say well we haven't really done exhaustive search yet and maybe we're looking in the wrong bands and and we've got the wrong devices and we wouldn't notice what an alien form was like to be so different to what we're used to but you know i'm not i don't really buy that that it shouldn't be as difficult as that like we i think we've searched enough there should be if it were everywhere if it was it should be everywhere we should see dyson's fears being put up sun's blinking in and out you know there should be a lot of evidence for those things and then there are other people argue well the sort of safari view of like well we're a primitive species still because we're not space faring yet and and and we're you know there's some kind of globe like universal rule not to interfere\n\

In [186]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n                                You are a helpful assistant.\n                                Answer ONLY from the provided transcript context.\n                                If the context is insufficient, just say you don't know.\n\n                                space age we should have heard a cacophony of voices we should have joined that cacophony of voices and what we did we opened our ears and we heard nothing and many people who argue that there are aliens would say well we haven't really done exhaustive search yet and maybe we're looking in the wrong bands and and we've got the wrong devices and we wouldn't notice what an alien form was like to be so different to what we're used to but you know i'm not i don't really buy that that it shouldn't be as difficult as that like we i think we've searched enough there should be if it were everywhere if it was it should be everywhere we should see dyson's fears being put up sun's blinking in and out you kn

## Step 4 - Generation

In [188]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of aliens is discussed in this video. The speaker shares their thoughts and opinions on the following points:

1. The possibility of alien civilizations and the likelihood of us being alone in the universe.
2. The argument that we haven't searched enough for alien life and that there should be evidence if it exists.
3. The idea that alien civilizations might communicate through thoughts or have a uniform way of communicating.
4. The possibility of a "violent dictatorship" among alien civilizations, where successful civilizations become more destructive.
5. The "great filter" theory, which suggests that something prevents civilizations from becoming multi-planetary or reaching out into the stars.
6. The idea that if we destroy ourselves, it's essential to consider how easy it is to do so.
7. The possibility of enhancing ourselves through technology, such as neural links, to become more advanced.

The speaker also shares their personal opinion that they think we are likely

## Building a Chain

In [190]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [191]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [192]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
    })

In [193]:
parallel_chain.invoke('who is Demis')

{'context': "to get world peace because there's also other corrupting things like wanting power over people and this kind of stuff which is not necessarily satisfied by by just abundance but i think it will help um and i think uh but i think ultimately ai is not going to be run by any one person or one organization i think it should belong to the world belong to humanity um and i think maybe many there'll be many ways this will happen and ultimately um everybody should have a say in that do you have advice for uh young people in high school and college maybe um if they're interested in ai or interested in having a big impact on the world what they should do to have a career they can be proud of her to have a life they can be proud of i love giving talks to the next generation what i say to them is actually two things i i think the most important things to learn about and to find out about when you're when you're young is what are your true passions is first of all there's two things on

In [194]:
parser = StrOutputParser()

In [195]:
main_chain = parallel_chain | prompt | llm | parser

In [196]:
main_chain.invoke('can you summarize the video')

'The speaker discusses explaining complex topics in physics and AI, and how it can be approached by starting with a deeper, simpler explanation that encompasses many mysteries of the universe. They mention that the standard model of physics does not work but is still being added to, and that a new explanation could give us insights into consciousness, life, and gravity.\n\nThey also talk about the connection between humans and AI, and how humans are already symbiotic with their devices. The speaker mentions Claude Shannon\'s first chess program in 1949 and Alan Turing\'s chess program, and how they were not powerful enough to run at the time.\n\nAdditionally, the speaker reflects on the moment when Deep Blue beat Garry Kasparov at chess, and how they were more impressed by Kasparov\'s mind than the machine\'s brute calculation power. The speaker believes that a human\'s ability to explain complex topics simply is a sign of intelligence, and that it\'s possible to enhance human intellig